# 02 · Configurações 1 e 2: modelo sem contexto × RAG simples
Mesmo modelo (`Qwen2.5-1.5B-Instruct`, rodando localmente na GPU do Colab), mesmo pedido de JSON.
A única diferença é o contexto recuperado da base.

In [ ]:
# Preparação do ambiente (Colab ou local)
import os, sys, subprocess
REPO_URL = "https://github.com/giuliano10/GS-GENENERATIVE-AI.git"
SUBPASTA = "assistente-acessibilidade-calcadas"   # pasta do projeto dentro do repositório

if "google.colab" in sys.modules:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    else:   # mesma sessão de outro notebook: atualiza o código
        subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
    os.chdir(os.path.join("/content/repo", SUBPASTA))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("Falha no pip install (detalhes acima)")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.getcwd())

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (use Ambiente de execução > Alterar tipo > T4 GPU)")

In [ ]:
from src.base_conhecimento import carregar_trechos
from src.avaliacao import carregar_jsonl
from src.busca import Buscador, carregar_limiar
from src.config import ARQ_TESTE
from src.gerador import carregar_modelo
from src.experimento import rodar
trechos = carregar_trechos(); teste = carregar_jsonl(ARQ_TESTE)
buscador = Buscador(trechos, modo="hibrido"); limiar = carregar_limiar()
modelo = carregar_modelo(com_lora=False)
print(modelo.nome, "| limiar", limiar)

## Configuração 1 — sem contexto

In [ ]:
sem_ctx = rodar("sem_contexto", teste, modelo)
for s in sem_ctx[:3]: print(s["pergunta"], "\n ->", s["bruto"][:300], "\n")

## Configuração 2 — RAG simples

In [ ]:
rag = rodar("rag", teste, modelo, buscador, limiar)
for s in rag[:3]: print(s["pergunta"], "\n ->", s["bruto"][:300], "\n")

In [ ]:
import pandas as pd
from src.avaliacao import corrigir, placar
ids = {t["id"] for t in trechos}; fids = {t["fonte_id"] for t in trechos}
pd.DataFrame([placar([corrigir(c, s, ids, fids) for c, s in zip(teste, saidas)], nome)
              for nome, saidas in [("sem_contexto", sem_ctx), ("rag", rag)]])